# Demonstration of Covariant Approximation Averaging (CAA) 

In [ ]:
import statpy as sp
import numpy as np
import matplotlib.pyplot as plt
from statpy.statistics import jackknife
from statpy.log import message

## Read correlator data

We begin by using `statpy` to load correlator data from a single ensemble with ID `9a`. The data has been **blinded** using a multiplicative blinding factor to prevent psychological bias during the analysis.

In [ ]:
# The 9a correlator data is not publicly redistributable and is therefore not
# included in this repository. Set DATA_DIR to the directory that holds it.
DATA_DIR = "data"
db = sp.database.core.DB(f"{DATA_DIR}/9a/fermionic.db")

# The conserved-current correlator is stored under a trailing "_r" tag denoting
# a specific lattice construction of the lc correlator; dropped here for brevity.
for tag in db.get_tags("_lc_r$"):
    db.rename_entry(tag, tag[:-2])

### Inspection of database

Next, we inspect the entries stored in the database:

In [ ]:
db.print_tags()

Each entry in the database is identified by a tag, composed of three parts:
1. Ensemble ID: here, always 9a
2. Solve type: indicates both the set of source positions and the numerical precision of the solve
3. Correlator type: specifies the quantum numbers used to construct the correlator from quark propagators

The **solve type** follows the pattern `X_Y`, where:
- The first label (`X`) refers to the set of source positions:
	- exact means a few source positions used for high-precision reference solves
	- sloppy refers to a different, larger set of source positions used for approximate solves
- The second label (`Y`) describes the numerical precision of the solve:
	- exact means a high-precision solution of the Dirac equation
	- sloppy means a reduced-precision (and thus faster) solve

In our specific case, we have the combinations:
- `exact_exact`: high-precision solve at the exact source position
- `exact_sloppy`: low-precision solve at the same exact source position (used to estimate bias)
- `sloppy_sloppy`: low-precision solves at different source positions (used for variance reduction)

### Entries

Each object associated with a tag is called an `entry`. An entry stores the **mean value** of the observable together with its corresponding [**jackknife sample**](https://en.wikipedia.org/wiki/Jackknife_resampling), which enables robust estimation of statistical uncertainties (via `db.jackknife_variance()`). 

Thanks to this structure, `statpy` makes it easy to compute **derived quantities** and **propagate statistical uncertainties** in a consistent and automated way through the `db.combine()` method.

A raw-data entry additionally retains the original per-configuration **measurement sample**. We use these raw samples during data preparation — for example, to average over source positions and spatial directions.

We can now inspect the entries in the database:

In [ ]:
entry = db.database["9a/exact_exact/G0_G0_lc"]
print("mean shape:  ", entry.mean.shape)
print("sample shape:", entry.sample.shape)
print("jks shape:   ", entry.jks.shape)
print("# configs:   ", len(entry.cfgs))

In [ ]:
print(db.database["9a/exact_exact/G0_G0_lc"].mean.shape)
print(db.database["9a/sloppy_sloppy/G0_G0_lc"].mean.shape)

Each row in the mean array corresponds to a different source position. In this case, the `exact_exact` tag includes a single source position, while the `sloppy_sloppy` tag includes eight source positions, reflecting the denser sampling used in the sloppy solves.

# Data preparation
For the current analysis, we are not interested in the `G5_G5_ll` leafs, which correspond to the [pion](https://de.wikipedia.org/wiki/Pion). To remove them from the database, we can do the following:

In [ ]:
pion_tags = db.get_tags("G5_G5_ll")
print(pion_tags)
for tag in pion_tags:
    db.remove_entry(tag)

# reprint db content
db.print_tags()

After removing the pion correlators, the remaining database entries are vector correlators of two types: `ll` (local-local) and `lc` (local-conserved). Both represent the same physical quantity but differ in how they are implemented on the lattice.

The `lc` version more closely resembles the structure of the continuum theory and has better symmetry properties, but it is also more expensive to compute. In contrast, the `ll` version is computationally cheaper but less faithful to continuum behavior.

Each of these vector correlators comes in three variants: `G0`, `G1`, and `G2`, corresponding to the three spatial directions. Due to the lattice’s rotational symmetry, these directions are physically equivalent. We can therefore average over them (separately for the ll and lc types) to reduce statistical noise and improve signal quality without losing physical information.

## Improved estimate
As described in the [README.md](https://github.com/spieseba/caa-control-variates/blob/main/README.md), we construct the **improved estimator** as:
$$
  O^\text{(imp)} = O^\text{(rest)} + O_G^\text{(appx)},
$$
where:
- $O^\text{(rest)} = O - O^\text{(appx)}$ is a bias correction
- $O_G^\text{(appx)} = \frac{1}{N_G} \sum_{g \in G} O^{\text{(appx)},g}$ is an average over $N_G$ symmetry transformations.

In this analysis, the `sloppy_sloppy` entires are used to estimate $O_G^\text{(appx)}$. For the `ll` correlator, we average over 128 source positions, while for the `lc` correlator, only 8 source positions are available:

In [ ]:
print(f'll sloppy_sloppy source positions: {db.database["9a/sloppy_sloppy/G0_G0_ll"].mean.shape[0]}')
print(f'lc sloppy_sloppy source positions: {db.database["9a/sloppy_sloppy/G0_G0_lc"].mean.shape[0]}')

The exact estimator $O$ is constructed from the exact_exact data, while its approximation $O^{(\text{appx})}$ is taken from the corresponding exact_sloppy data. Both are computed at the same single source position on each configuration to ensure strong correlations, which are crucial for reducing the variance of the bias correction term $O^{(\text{rest})}$:

In [ ]:
print(f'll exact_exact source positions: {db.database["9a/exact_exact/G0_G0_ll"].mean.shape[0]}')
print(f'll exact_sloppy source positions: {db.database["9a/exact_sloppy/G0_G0_ll"].mean.shape[0]}')

In the context of [arXiv 2410.17053](https://inspirehep.net/literature/2841842), we are particularly interested in constructing improved estimators for:
- the `ll` correlator $C^{(\text{ll})}$
- the `lc` correlator $C^{(\text{lc})}$, and
- their ratio, defined as $Z(t) = C^{(\text{lc})}/C^{(\text{ll})}$.

### Source averaging

When analyzing the `ll` and `lc` correlators individually, the improved approximation term $O_G^{(\text{appx})}$ can be constructed by averaging over all available sloppy_sloppy source positions.

However, for the ratio $Z(t)$, it is beneficial to restrict the averaging to the common source positions used in both `ll` and `lc` estimates. This ensures that the numerator and denominator are highly correlated, which leads to partial cancellation of fluctuations in the ratio’s statistical error.

This is captured by the standard variance propagation formula:
$$
    \sigma_f^2 \approx f^2 \left[\left(\frac{\sigma_A}{A}\right)^2 + \left(\frac{\sigma_B}{B}\right)^2 - 2\frac{\sigma_{AB}}{AB} \right], \quad \text{where } f = \frac{A}{B}.
$$
The last term accounts for the covariance between numerator and denominator. When A and B are strongly correlated, this term significantly reduces the overall variance, leading to a more precise estimate of $Z(t)$. 

In our case, the set of source positions used for the `lc` correlator is a subset of those used for the `ll` correlator. To exploit the correlation between numerator and denominator in the ratio $Z(t)$, we construct three separate improved estimators:
1. A full source average for the `ll` correlator over all available sloppy_sloppy source positions.
2. A full source average for the `lc` correlator over its (fewer) available source positions.
3. An additional `ll` source average, restricted to the same source positions used for the `lc` correlator, to serve as a matched denominator in the ratio $Z(t)$.

In the following code blocks, we perform this source-matched averaging and remove the original, unaveraged data from the database to keep the dataset clean and focused.

In [ ]:
for i in range(3):
    src_tag_lc = f"9a/sloppy_sloppy/G{i}_G{i}_lc"
    src_tag_ll = f"9a/sloppy_sloppy/G{i}_G{i}_ll"
    lc = db.database[src_tag_lc]
    ll = db.database[src_tag_ll]
    # v2 sample is a (n_cfgs, n_src, n_t) array; index cfgs by position.
    ll_pos = {cfg: k for k, cfg in enumerate(ll.cfgs)}
    cfgs = list(lc.cfgs)
    rows = []
    for cfg in cfgs:
        matched_idxs = [k for k, pt in enumerate(ll.misc["ptsrcs"][cfg])
                        if pt in lc.misc["ptsrcs"][cfg]]
        rows.append(np.mean(ll.sample[ll_pos[cfg]][matched_idxs], axis=0).real)
    db.add_entry(f"9a/sloppy_sloppy/G{i}_G{i}_ll/matched",
                 sample=np.array(rows), weights=np.ones(len(cfgs)), cfgs=np.array(cfgs))

In [ ]:
# average over the source axis of each raw entry (in place)
for i in range(3):
    for ctype in ["ll", "lc"]:
        for dataset in ["exact_exact", "exact_sloppy", "sloppy_sloppy"]:
            tag = f"9a/{dataset}/G{i}_G{i}_{ctype}"
            e = db.database[tag]
            avg_sample = np.mean(e.sample, axis=1).real
            db.remove_entry(tag)
            db.add_entry(tag, sample=avg_sample, weights=e.weights, cfgs=e.cfgs)

In [ ]:
db.print_tags()

### Spatial index averaging
As discussed above, we exploit the rotational symmetry of the lattice to average the correlators over the spatial directions. We again remove the original, unaveraged correlators associated with individual spatial indices:

In [ ]:
def average_i(tags):
    # average the samples of the G0/G1/G2 entries cfg-wise -> Gi_Gi
    entries = [db.database[t] for t in tags]
    avg_sample = np.mean([e.sample for e in entries], axis=0).real
    dst_tag = tags[0].replace("G0_G0", "Gi_Gi")
    db.add_entry(dst_tag, sample=avg_sample, weights=entries[0].weights, cfgs=entries[0].cfgs)
    for tag in tags:
        db.remove_entry(tag)

In [ ]:
for dataset in ["exact_exact", "exact_sloppy", "sloppy_sloppy"]:
    for ctype in ["ll", "lc"]:
        average_i([f"9a/{dataset}/G{i}_G{i}_{ctype}" for i in range(3)])
        if dataset == "sloppy_sloppy" and ctype == "ll":
            average_i([f"9a/{dataset}/G{i}_G{i}_{ctype}/matched" for i in range(3)])

In [ ]:
db.print_tags()

# Data exploration and checks
Before applying CAA to the correlator data, we first explore and validate the dataset. This includes basic sanity checks and visualizations to ensure the data is well-behaved and suitable for further analysis.

### Plot Markov histories and check thermalization
We begin by verifying that the correlator data, generated through a Markov Chain Monte Carlo (MCMC) simulation, are thermalized, i.e., the Markov chain has reached equilibrium and samples the correct target distribution. As a simple check, we plot the correlator values at selected time slices against the Monte Carlo time $t_\text{MC}$.

In [ ]:
cfg_idxs = [int(cfg.split("-")[-1]) for cfg in db.database["9a/exact_exact/Gi_Gi_ll"].cfgs]

Gi_Gi_ll_sloppy = db.database["9a/exact_sloppy/Gi_Gi_ll"].sample
Gi_Gi_lc_sloppy = db.database["9a/exact_sloppy/Gi_Gi_lc"].sample
Gi_Gi_ll_exact = db.database["9a/exact_exact/Gi_Gi_ll"].sample
Gi_Gi_lc_exact = db.database["9a/exact_exact/Gi_Gi_lc"].sample

fig, (ax0, ax1) = plt.subplots(ncols=2, figsize=(16,6))

ax0.set_title(r"$C^\text{ll}(n_t)$")
ax0.set_xlabel(r"Monte Carlo time $t_{MC}$")
ax0.set_ylabel(r"$C^\text{ll}(n_t)$")
for t in [1,2,4,6]:
    ax0.plot(cfg_idxs, Gi_Gi_ll_exact[:,t].real, marker="+", ls="", label=r"exact_exact, $n_t=$" + f"{t}")
    ax0.plot(cfg_idxs, Gi_Gi_ll_sloppy[:,t].real, marker="x", ls="", label=r"exact_sloppy, $n_t=$" + f"{t}")
ax0.legend(loc="center right")
ax0.grid()

ax1.set_title(r"$C^\text{lc}(n_t)$")
ax1.set_xlabel(r"Monte Carlo time $t_{MC}$")
ax1.set_ylabel(r"$C^\text{lc}(n_t)$")
for t in [1,2,4,6]:
    ax1.plot(cfg_idxs, Gi_Gi_lc_exact[:,t].real, marker="+", ls="", label=r"exact_exact, $n_t=$" + f"{t}")
    ax1.plot(cfg_idxs, Gi_Gi_lc_sloppy[:,t].real, marker="x", ls="", label=r"exact_sloppy, $n_t=$" + f"{t}")
ax1.legend(loc="center right")
ax1.grid()

plt.suptitle("Markov histories - 9a")
plt.tight_layout()
plt.show()

Since the data fluctuates around a stable mean without exhibiting visible trends or drifts, we conclude that the observables are thermalized over the given Monte Carlo time range.

### Treating autocorrelations

Since the correlator data originate from a Markov Chain Monte Carlo (MCMC) simulation, it is essential to check for autocorrelations between successive configurations. Autocorrelations can artificially reduce the estimated statistical error if not properly accounted for. To assess this, we performed a binning study for an exemplary time slice ($n_t = 3$) and found that the variance remains stable across increasing bin sizes. This indicates that autocorrelations are negligible in our dataset, and we can proceed with the analysis using the raw, unbinned samples.

In [ ]:
def Gi_Gi_binning_study(tag, nt, binsizes=[1,2,4,8]):
    e = db.database[tag]
    var = {}
    for b in binsizes:
        jks = e.jks if b == 1 else jackknife.delayed_binning(e.jks, b, weights=e.weights)
        var[b] = jackknife.variance(jks)
    message(f"Delayed binning study of {tag} (nt = {nt}): var (binsizes={binsizes}): \n\t\t\t\t{[float(var[b][nt]**.5) for b in binsizes]}")

message("--------------------- Binning study for ensemble 9a --------------------")
for dataset in ["exact_exact", "exact_sloppy", "sloppy_sloppy"]:
    for ctype in ["ll", "lc"]:
        message(f"--- {dataset} - {ctype} ---")
        Gi_Gi_binning_study(f"9a/{dataset}/Gi_Gi_{ctype}", 3, [1,2,4,8])
message("-------------------------------------------------------\n")

### Correlator consistency
Next, we compare the correlator means for different solve types (`exact_exact`, `exact_sloppy`, `sloppy_sloppy`) to ensure that the sloppy solves are reliable approximations of the exact ones.

To do this, we analyze the dimensionless combination $t^3 C^\text{lc}(t)$, which helps highlight deviations in the correlator behavior across time slices. This form is often used in studies of the hadronic vacuum polarization contribution to the muon $g\!-\!2$, as it emphasizes the region most relevant to the physical observable. Here, it simply serves as a useful diagnostic tool.

In [ ]:
def add_solve_type_to_plot(ax, solve_type, ctype="ll", ensemble="9a"):
    Gi_Gi = db.database[f"{ensemble}/{solve_type}/Gi_Gi_{ctype}"].mean
    ts = np.arange(len(Gi_Gi))
    Gi_Gi_var = db.jackknife_variance(f"{ensemble}/{solve_type}/Gi_Gi_{ctype}")
    m = {"exact_exact": "+", "exact_sloppy": "x", "sloppy_sloppy": "."}[solve_type]
    ax.errorbar(ts, Gi_Gi * ts**3, Gi_Gi_var**.5 * ts**3, marker=m, capsize=4, linestyle="", label=solve_type)


fig, ax = plt.subplots(figsize=(16,5))

add_solve_type_to_plot(ax, "exact_exact")
add_solve_type_to_plot(ax, "exact_sloppy")
add_solve_type_to_plot(ax, "sloppy_sloppy")

ax.set_ylabel(r"$t^3 C^\text{lc}(t)$")
ax.set_xlabel(r"$t/a$")
ax.set_xlim(0.0,20.5)
ax.set_ylim(0.0, 0.0875)
ax.legend()
ax.grid()

plt.tight_layout()
plt.show()

We observe that `exact_exact` and `exact_sloppy` correlators align very well across all time slices, with comparable statistical errors. This confirms that sloppy solves provide a high-quality approximation when computed at the same source positions as the exact solves.

In contrast, the sloppy_sloppy correlator agrees with the exact_sloppy result only within statistical margins. A mild tension is visible in the range $t = 8\text{–}11$, likely a statistical fluctuation specific to this dataset. This discrepancy may be due to averaging over different source positions, which can introduce additional variance in certain time regions.

The statistical errors for `sloppy_sloppy` are significantly smaller (reduced by roughly a factor of $\sim \sqrt{8}$) due to averaging over eight source positions instead of just one.

## Application of CAA
Let us now apply CAA. For this, we define a helper function that allows us to specify the tags for $O$, $O^\text{(appx)}$, and $O^\text{(appx)}_G$ from which the improved estimator:
$$
    O^\text{(imp)} = O^\text{(rest)} + O_G^\text{(appx)}, \quad\text{with } O^\text{(rest)} = O - O^\text{(appx)} 
$$
is computed.

In [ ]:
def apply_CAA(tag, appx_tag, appx_G_tag, CAA_label="CAA"):
    CAA_tag = f'{tag.split("/")[0]}/{CAA_label}/{tag.split("/")[-1]}'
    db.combine(tag, appx_tag, f=lambda obs,obs_appx: obs - obs_appx, dst_tag=f"{CAA_tag}_rest")
    db.combine(f"{CAA_tag}_rest", appx_G_tag, f=lambda obs_rest,obs_appx_G: obs_rest + obs_appx_G, dst_tag=CAA_tag)

### $C^\text{(ll)}$

We start with the improved estimator for $C^\text{(ll)}$. As mentioned above, we use the `exact_exact` $C^\text{(ll)}$ data for $O$, the `exact_sloppy` $C^\text{(ll)}$ data for $O^\text{appx}$, and `sloppy_sloppy` $C^\text{(ll)}$ data for $O_G^\text{appx}$:

In [ ]:
apply_CAA("9a/exact_exact/Gi_Gi_ll", "9a/exact_sloppy/Gi_Gi_ll", "9a/sloppy_sloppy/Gi_Gi_ll")

We can now compare the statistical errors of the improved estimate with the `exact_exact` and `sloppy_sloppy` estimates:

In [ ]:
print((db.jackknife_variance("9a/exact_exact/Gi_Gi_ll") / db.jackknife_variance("9a/CAA/Gi_Gi_ll"))**.5)

In [ ]:
print((db.jackknife_variance("9a/sloppy_sloppy/Gi_Gi_ll") / db.jackknife_variance("9a/CAA/Gi_Gi_ll"))**.5)

As expected, the error of the improved estimate is roughly reduced to the same order of magnitude as for the `sloppy_sloppy` data, which can also be verified visually:

In [ ]:
def add_CAA_to_plot(ax, tag, appx_tag, appx_G_tag, CAA_tag):
    obs = db.database[tag].mean
    obs_var = db.jackknife_variance(tag)
    obs_appx = db.database[appx_tag].mean
    obs_appx_var = db.jackknife_variance(appx_tag)
    obs_G_appx = db.database[appx_G_tag].mean
    obs_G_appx_var = db.jackknife_variance(appx_G_tag)
    obs_CAA = db.database[CAA_tag].mean
    obs_CAA_var = db.jackknife_variance(CAA_tag)

    ts = np.arange(len(obs))

    ax.errorbar(ts, ts**3 * obs, ts**3 * obs_var**.5, marker="+", capsize=3, linestyle="", label=r"$O$")
    ax.errorbar(ts, ts**3 * obs_appx, ts**3 * obs_appx_var**.5, marker="x", capsize=3, linestyle="", label=r"$O^\text{(appx)}$")
    ax.errorbar(ts, ts**3 * obs_G_appx, ts**3 * obs_G_appx_var**.5, marker="x", capsize=3, linestyle="", label=r"$O^\text{(appx)}_G$")
    ax.errorbar(ts, ts**3 * obs_CAA, ts**3 * obs_CAA_var**.5, marker="+", capsize=3, linestyle="", label=r"$O^\text{(imp)}$")

    ax.set_ylabel(r"$t^3 C^\text{lc}(t)$")
    ax.set_xlabel(r"$t/a$")


fig, ax = plt.subplots(ncols=1, figsize=(16,5))

add_CAA_to_plot(ax, tag="9a/exact_exact/Gi_Gi_ll", appx_tag="9a/exact_sloppy/Gi_Gi_ll", appx_G_tag="9a/sloppy_sloppy/Gi_Gi_ll", CAA_tag="9a/CAA/Gi_Gi_ll")

ax.set_xlim(0.5,6.5)
ax.set_ylim(0.056, 0.085)

ax.set_title(r"CAA for $C^\text{(ll)}(t)$")
ax.grid()
ax.legend()

plt.tight_layout()
plt.show()

### $C^\text{(lc)}$
Analogous to the `ll` case, we could directly use the `sloppy` solves of the `lc` correlator to construct an improved estimator. However, since `ll` and `lc` correlators describe the same physical quantity (up to discretization effects) and are highly correlated, we can alternatively treat the `ll` correlator as a cheap proxy for `lc` and use it to build an improved sloppy estimate for $C^{\text{(lc)}}$. To estimate the bias, we use matched source positions between `ll` and `lc`, ensuring high correlation between the observables.

Once we have this improved sloppy estimator, we use it to construct a CAA-improved version of the exact $C^{\text{(lc)}}$ correlator:

In [ ]:
apply_CAA(tag="9a/sloppy_sloppy/Gi_Gi_lc", appx_tag="9a/sloppy_sloppy/Gi_Gi_ll/matched", appx_G_tag="9a/sloppy_sloppy/Gi_Gi_ll", CAA_label="CAA_sloppy")
apply_CAA(tag="9a/exact_exact/Gi_Gi_lc", appx_tag="9a/exact_sloppy/Gi_Gi_lc", appx_G_tag="9a/CAA_sloppy/Gi_Gi_lc", CAA_label="CAA")

Let us compare the statistical errors of the improved estimate with the `exact_exact` `lc` and `sloppy_sloppy` `ll` estimates:

In [ ]:
print((db.jackknife_variance("9a/exact_exact/Gi_Gi_lc") / db.jackknife_variance("9a/CAA/Gi_Gi_lc"))**.5)

In [ ]:
print((db.jackknife_variance("9a/sloppy_sloppy/Gi_Gi_ll") / db.jackknife_variance("9a/CAA/Gi_Gi_lc"))**.5)

The statistical error of the improved estimator is within 20% of that of the `sloppy_sloppy` `ll` data. 

Additionally, we compare to a naive CAA estimate that does not incorporate the `ll` correlator data:

In [ ]:
apply_CAA(tag="9a/exact_exact/Gi_Gi_lc", appx_tag="9a/exact_sloppy/Gi_Gi_lc", appx_G_tag="9a/sloppy_sloppy/Gi_Gi_lc", CAA_label="CAA_naive")
print((db.jackknife_variance("9a/CAA/Gi_Gi_lc") / db.jackknife_variance("9a/CAA_naive/Gi_Gi_lc"))**.5)

Thus, employing a two-stage approach, where the sloppy `lc` data are first improved using the `ll` correlator, yields a roughly twofold reduction in statistical error across most time slices, compared to using the `lc` data alone.

In [ ]:
fig, (ax0, ax1) = plt.subplots(ncols=2, figsize=(16,6))

add_CAA_to_plot(ax0, tag="9a/sloppy_sloppy/Gi_Gi_lc", appx_tag="9a/sloppy_sloppy/Gi_Gi_ll/matched", appx_G_tag="9a/sloppy_sloppy/Gi_Gi_ll", CAA_tag="9a/CAA_sloppy/Gi_Gi_lc")
ax0.set_xlim(0.5,10.5)
ax0.set_ylim(0.033, 0.0875)
ax0.set_title(r"CAA for $C^\text{(lc)}_\text{sloppy}(t)$")
ax0.grid()
ax0.legend()

add_CAA_to_plot(ax1, tag="9a/sloppy_sloppy/Gi_Gi_lc", appx_tag="9a/sloppy_sloppy/Gi_Gi_ll/matched", appx_G_tag="9a/sloppy_sloppy/Gi_Gi_ll", CAA_tag="9a/CAA_sloppy/Gi_Gi_lc")
ax1.set_xlim(3.5,10.5)
ax1.set_ylim(0.033, 0.044)
ax1.set_title(r"Zoom")
ax1.grid()
#ax1.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, figsize=(16,5))

add_CAA_to_plot(ax, tag="9a/exact_exact/Gi_Gi_lc", appx_tag="9a/exact_sloppy/Gi_Gi_lc", appx_G_tag="9a/CAA_sloppy/Gi_Gi_lc", CAA_tag="9a/CAA/Gi_Gi_lc")

ax.set_xlim(0.5,6.5)
ax.set_ylim(0.033, 0.0475)

ax.set_title(r"CAA for $C^\text{(lc)}(t)$")
ax.grid()
ax.legend()

plt.tight_layout()
plt.show()

### $Z(t) = C^{(\text{lc})}/C^{(\text{ll})}$
Finally, we are interested in computing an improved estimate for the ratio $Z(t) = C^{(\text{lc})}/C^{(\text{ll})}$.

To this end, we compare three different combinations of lc and ll estimators to assess which provides the most precise result. The underlying idea is that the statistical error of a ratio is suppressed when numerator and denominator exhibit similar fluctuations. The tested strategies are:

1. **Baseline**: Use `exact_exact` estimates for both $C^{(\text{lc})}$ and $C^{(\text{ll})}$.

In [ ]:
db.combine("9a/exact_exact/Gi_Gi_lc", "9a/exact_exact/Gi_Gi_ll", f=lambda lc,ll: lc/ll, dst_tag="9a/exact_exact/Zt")

2. **Full CAA**: Use the improved estimators from the full CAA procedure for both $C^{(\text{lc})}$ and $C^{(\text{ll})}$,  incorporating all available point sources.

In [ ]:
db.combine("9a/CAA/Gi_Gi_lc", "9a/CAA/Gi_Gi_ll", f=lambda lc,ll: lc/ll, dst_tag="9a/CAA/Zt")

3. **Matched-source CAA**: Compute a CAA estimate of $C^{(\text{lc})}$ that relies only on `lc` data, and a CAA estimate of $C^{(\text{ll})}$ constructed from only `ll` data on the same set of source positions as `lc`.

In [ ]:
apply_CAA(tag="9a/exact_exact/Gi_Gi_lc", appx_tag="9a/exact_sloppy/Gi_Gi_lc", appx_G_tag="9a/sloppy_sloppy/Gi_Gi_lc", CAA_label="CAA_exact_sloppy")
apply_CAA(tag="9a/exact_exact/Gi_Gi_ll", appx_tag="9a/exact_sloppy/Gi_Gi_ll", appx_G_tag="9a/sloppy_sloppy/Gi_Gi_ll/matched", CAA_label="CAA_matched")
db.combine("9a/CAA_exact_sloppy/Gi_Gi_lc", "9a/CAA_matched/Gi_Gi_ll", f=lambda lc,ll: lc/ll, dst_tag="9a/CAA_matched/Zt")

In [ ]:
Zt_exact_exact = db.database["9a/exact_exact/Zt"].mean
Zt_exact_exact_var = db.jackknife_variance("9a/exact_exact/Zt")
Zt_CAA = db.database["9a/CAA/Zt"].mean
Zt_CAA_var = db.jackknife_variance("9a/CAA/Zt")
Zt_CAA_matched = db.database["9a/CAA_matched/Zt"].mean
Zt_CAA_matched_var = db.jackknife_variance("9a/CAA_matched/Zt")
ts = np.arange(len(Zt_exact_exact))

fig, ax = plt.subplots(figsize=(14,6))

ax.errorbar(ts, Zt_exact_exact, Zt_exact_exact_var**.5, capsize=3, marker="+", linestyle="", label="exact_exact")
ax.errorbar(ts, Zt_CAA, Zt_CAA_var**.5, capsize=3, marker="+", linestyle="", label="CAA", color="C2")
ax.errorbar(ts, Zt_CAA_matched, Zt_CAA_matched_var**.5, capsize=3, marker="+", linestyle="", label="CAA matched", color="C3")

ax.set_ylabel(r"$Z(t)$")
ax.set_xlabel(r"$t/a$")

ax.set_ylim(0.65, 0.8)
ax.set_xlim(3.5,20)

ax.grid()
ax.legend()
plt.tight_layout()
plt.show()

Relative to the `exact_exact` baseline, the **full CAA** estimate performs significantly worse due to reduced correlations between numerator and denominator, which leads to larger statistical uncertainty in the ratio. 

In contrast, the **matched-source CAA** approach, where both $C^{(\text{lc})}$ and $C^{(\text{ll})}$ are constructed from the same set of source positions, exploits strong correlations between the two and achieves smaller errors than the baseline. This highlights the importance of matching source positions when forming ratios of correlated observables.

In [ ]:
print(Zt_exact_exact_var**.5 / Zt_CAA_var**.5)

In [ ]:
print(Zt_exact_exact_var**.5 / Zt_CAA_matched_var**.5)